## 2.Build The Data Collector

#### Import Libraries

In [1]:
import requests
from bs4 import BeautifulSoup
import csv
from datetime import datetime

#The User Agent tells the browser we are a standard Windows Chrome Browser, not a python script
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept-Language': 'en-US, en;q=0.5'
}

#### Create the Extraction Function

In [2]:
def check_price(url):
    #1 fetch the webpage HTML
    response=requests.get(url,headers=HEADERS)

    soup = BeautifulSoup(response.content, 'html.parser')

    # 1. Extract Title (Try the two most common Amazon title tags)
    title_element = soup.find(id='productTitle') or soup.find(class_='product-title-word-break')
    title = title_element.get_text().strip() if title_element else "Unknown Product"

    # 2. List of common Amazon price class names
    price_classes = [
        'a-price-whole',           # Standard layout
        'a-color-price',           # Often used for deals or out-of-stock alternative pricing
        'apexPriceToPay',          # Used in some electronics categories
        'priceToPay',              # Alternate standard tag
        'a-size-medium a-color-price' # Sometimes used in the "No-JS" raw HTML
    ]
    
    price_str = None
    
    # 3. Iterate through the list until we find a match
    for p_class in price_classes:
        price_element = soup.find('span', class_=p_class) or soup.find('div', class_=p_class)
        if price_element:
            price_str = price_element.get_text().strip()
            break # Stop looking once we find the first valid price
            
    # 4. Clean the text and convert to integer
    if price_str:
        # Remove the Rupee symbol, commas, and any trailing decimals
        clean_price = price_str.replace('₹', '').replace(',', '').replace('.00', '').replace('.', '').strip()
        
        try:
            price = int(clean_price)
            return {"title": title, "price": price, "timestamp": datetime.now()}
        except ValueError:
            print(f"Error: Found price text '{price_str}', but could not convert '{clean_price}' to an integer.")
            return None
    else:
        print("Error: Could not find the price using any of the known HTML classes.")
        return None

#### Log the results to a CSV

In [3]:
def log_data(data, filename="price_history.csv"):
    if not data:
        return
    #Open the file in append mode so we don't overwrite yesterday's data
    with open(filename,mode='a',newline='',encoding='utf-8') as file:
        writer=csv.writer(file)
        #Write: Title, Price, Date, Time
        writer.writerow([data['title'],data['price'],data['timestamp'].date(),data['timestamp'].time()])
    
    print(f"Logged: {data['title']} at ₹{data['price']}")

#Test the pipeline
test_url = "https://www.amazon.in/dp/B08N5W4NNB"
result = check_price(test_url)
log_data(result)
    

Logged: Apple MacBook Air Laptop: Apple M1 chip, 13.3-inch/33.74 cm Retina Display, 8GB RAM, 256GB SSD Storage, Backlit Keyboard, FaceTime HD Camera, Touch ID. Works with iPhone/iPad; Space Grey at ₹67990


## 3.Engineer the features

#### Generate the synthetic dataset

In [4]:
import pandas as pd
import random
from datetime import timedelta, date

#Generate 90 days of dates looking back from today
dates=[date.today()-timedelta(days=x) for x in range(90,0,-1)]
base_price=80000
prices=[]

for _ in dates:
    #Simulate normal pricing 85% of the time, and sudden price drops 15% of the time
    if random.random()>0.85:
        prices.append(base_price-random.randint(3000,8000)) #Sale drop
    else:
        prices.append(base_price+random.randint(-500,500)) #Minor fluctuations

#Create the Dataframe and save it
df=pd.DataFrame({'date':pd.to_datetime(dates),'title':'MacBook','price':prices})
df.to_csv('price_history.csv',index=False)
print("Synthetic dataset created: price_history.csv")


Synthetic dataset created: price_history.csv


#### Build the pandas pipeline

In [5]:
#1.Ingest the data
df=pd.read_csv('price_history.csv',parse_dates=['date'])

#2.Engineer the features
df['day_of_week']=df['date'].dt.dayofweek #Monday=0 Sunday=6
df['is_weekend']=df['day_of_week'].apply(lambda x:1 if x>=5 else 0)

#3.Engineer the price momentum features
#calculate the 7 day average to see if today's price is hsitorically good
df['rolling_avg_7d']=df['price'].rolling(window=7).mean()
df['pct_diff_from_avg']=((df['price']-df['rolling_avg_7d'])/df['rolling_avg_7d'])*100

#4.Create the target variable
#This is what the model will try to predict
#We shift the price column backward by 1 to peek at tomorrow's price
df['next_day_price']=df['price'].shift(-1)
#1 means price drops tomorrow 0 means price stays the same
df['drop_tomorrow']=(df['next_day_price']<df['price']).astype(int)

#Clean the pipeline by dropping the rows when NaN values which were formed by rolling and shifting
df=df.dropna()

print(df[['date','price','rolling_avg_7d','pct_diff_from_avg','drop_tomorrow']].tail(10))

         date  price  rolling_avg_7d  pct_diff_from_avg  drop_tomorrow
79 2026-10-03  79636    79354.714286           0.354466              1
80 2026-10-04  74792    78582.714286          -4.823853              0
81 2026-10-05  79669    78546.142857           1.429551              0
82 2026-10-06  80162    78542.571429           2.061848              1
83 2026-10-07  80020    78556.714286           1.862712              0
84 2026-10-08  80195    78638.000000           1.979959              1
85 2026-10-09  79854    79189.714286           0.838854              0
86 2026-10-10  80350    79291.714286           1.334674              1
87 2026-10-11  79561    79973.000000          -0.515174              0
88 2026-10-12  80005    80021.000000          -0.019995              1


## 4.Train the Prediction Brain 

In [31]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

features=['day_of_week', 'is_weekend', 'rolling_avg_7d', 'pct_diff_from_avg']
X=df[features]
y=df['drop_tomorrow']

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

model=RandomForestClassifier()
model.fit(X_train,y_train)

predictions=model.predict(X_test)

acc_score=accuracy_score(y_test,predictions)
print(f"Model Accuracy: {acc_score * 100:.2f}%\n")

print("Detailed Performance:")
print(classification_report(y_test, predictions, zero_division=0))

Model Accuracy: 52.94%

Detailed Performance:
              precision    recall  f1-score   support

           0       0.50      0.50      0.50         8
           1       0.56      0.56      0.56         9

    accuracy                           0.53        17
   macro avg       0.53      0.53      0.53        17
weighted avg       0.53      0.53      0.53        17



In [32]:
import joblib

# This saves your trained model to your project folder
joblib.dump(model, 'deal_model.pkl')

['deal_model.pkl']